# 闭包、嵌套与高阶函数

嵌套函数、闭包、高阶函数是层层递进的三个概念：**嵌套是语法基础，闭包是嵌套产生的一种「函数 + 环境」对象，高阶函数是把函数当值来用的编程方式**，而装饰器正是三者结合的产物。理解了这条主线，`sorted(key=...)`、回调、装饰器这些日常代码就都能看透了。

## 一、嵌套函数：函数里定义函数

`def` 是可执行语句，所以函数可以定义在任何地方——包括另一个函数的内部。这样做有两个直接好处：

- **外部访问不到它**，天然适合封装只服务于外层的辅助逻辑；
- 它可以**读取**外层函数的局部变量，作用域查找遵循 LEGB 规则：Local（自己）→ Enclosing（外层函数）→ Global（模块）→ Builtin（内建）。

In [1]:
def outer():
    x = 10
    def inner():
        return x + 1        # 自己没有 x，沿 LEGB 向外层找到 x
    return inner()          # 在 outer 内部调用

print(outer())  # 11

11


嵌套最常见的实战场景之一：外层做一次性的准备工作（参数校验、数据准备），内层专心做递归等主逻辑。比如二分查找的递归部分，用嵌套函数封装后就不用把 `sorted_list` 一层层传下去。

In [2]:
def binary_search(sorted_list, target):
    if not sorted_list:
        return -1

    def search(lo, hi):                     # 递归辅助函数，外部看不到
        if lo > hi:
            return -1
        mid = (lo + hi) // 2
        if sorted_list[mid] == target:
            return mid
        return search(mid + 1, hi) if sorted_list[mid] < target else search(lo, mid - 1)

    return search(0, len(sorted_list) - 1)

print(binary_search([1, 3, 5, 7, 9], 7))    # 3
print(binary_search([1, 3, 5, 7, 9], 4))    # -1
print(binary_search([], 1))                 # -1：外层的准备工作挡住空列表

3
-1
-1


## 二、闭包：内层函数「带走了」外层变量

上一节的 `inner` 是在 `outer` 里被调用的，还没什么稀奇。闭包的关键在于：**内层函数被当作返回值送出去，而它引用的外层变量也跟着一起被「打包带走」**。下面 `multiply` 里的 `factor` 既不是它自己的参数也不是局部变量，而是**自由变量**——按常理 `make_multiplier` 返回后局部变量就该被回收了，但闭包让 `factor` 继续活着。

**闭包 = 内层函数 + 它捕获的自由变量。**

In [1]:
def make_multiplier(factor):
    def multiply(n):
        return n * factor       # factor 是自由变量
    return multiply             # 不加括号：返回函数本身，而不是调用结果

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(10), triple(10))   # 20 30

# 函数返回后 factor 依然活着，存放在 __closure__ 的 cell 单元格里
print(double.__closure__[0].cell_contents)   # 2
print(triple.__closure__[0].cell_contents)   # 3

20 30
2
3


`__closure__` 有两个细节值得注意：

- **每次调用 `make_multiplier` 都产生一个独立的新环境**，`double` 和 `triple` 各存各的 `factor`，互不干扰；
- 只有捕获了自由变量的函数才有非空的 `__closure__`，普通函数的该属性是 `None`。

这种「用一个函数定制出另一个函数」的写法叫**工厂函数**，是闭包最典型的用途。

## 三、nonlocal：让闭包持有可变状态

闭包捕获的变量默认只能**读**。要在内层函数里给它赋值，必须先用 `nonlocal` 声明；否则 `count += 1` 会让 Python 把 `count` 当作内层的新局部变量（**赋值语句决定作用域**），一读就触发 `UnboundLocalError`。

有了 `nonlocal`，闭包就成了**带私有状态的函数**——比定义一个类轻量得多。

In [4]:
def make_counter():
    count = 0
    def inc():
        nonlocal count          # 去掉这行，下面就会 UnboundLocalError
        count += 1
        return count
    return inc

c1 = make_counter()
c2 = make_counter()
print(c1(), c1(), c1())         # 1 2 3：c1 有自己的记忆
print(c2())                     # 1：每个闭包的环境互不影响

1 2 3
1


## 四、高阶函数：把函数当值来用

Python 中函数是**一等公民**：可以赋值给变量、存进容器、当参数传、当返回值返回。**接收函数为参数，或返回函数的函数，就叫高阶函数**。

它带来的核心能力是：把「策略」从「流程」里抽出来，同一段流程就能复用于不同需求。

In [5]:
def shout(s):
    return s.upper() + "!"

def whisper(s):
    return s.lower() + "..."

# 函数可以赋值给变量（注意函数名不加括号）
speak = shout
print(speak("hello"))           # HELLO!

# 也可以存进字典，用来做「策略分派」
styles = {"loud": shout, "quiet": whisper}
for name, fn in styles.items():
    print(name, "->", fn("Hello"))

HELLO!
loud -> HELLO!
quiet -> hello...


In [6]:
nums = [3, 1, -2, -5, 4]

# sorted 的 key 参数接收一个函数，决定「按什么排」
print(sorted(nums, key=abs))                 # [1, -2, 3, 4, -5]

print(list(map(str, [1, 2, 3])))             # ['1', '2', '3']
print(list(filter(None, [0, 1, '', 'a'])))   # [1, 'a']：只保留真值

[1, -2, 3, 4, -5]
['1', '2', '3']
[1, 'a']


`sorted(key=...)`、`max`、`min`、`map`、`filter` 都在接收函数——你其实一直在用高阶函数。「返回函数」的方向同样常见，`make_multiplier` 就是一例；两个方向结合起来，就是下一节的装饰器。

## 五、装饰器：三者的结合

装饰器是**接收一个函数、返回一个新函数**的高阶函数，而新函数通常还是个闭包，把被装饰的函数和附加行为一起捕获进来。

`@deco` 只是语法糖，等价于先定义 `f`，再执行 `f = deco(f)`。

In [9]:
def log_calls(func):                # 高阶：接收函数
    def wrapper(*args, **kwargs):   # 闭包：捕获 func
        print(f"调用 {func.__name__}，参数 {args} {kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} 返回 {result}")
        return result
    return wrapper                  # 高阶：返回新函数

@log_calls                          # 等价于 add = log_calls(add)
def add(a, b):
    return a + b

print(add(2, 3))

调用 add，参数 (2, 3) {}
add 返回 5
5


装饰器本身还可以带参数（如 `@retry(3)`）。这时要多包一层：最外层先接收配置，返回真正的装饰器——一共三层嵌套，正好把前面的三个概念全部用上。

In [8]:
def retry(times):                           # 第 1 层：接收配置
    def deco(func):                         # 第 2 层：接收函数（真正的装饰器）
        def wrapper(*args, **kwargs):       # 第 3 层：闭包同时捕获 func 和 times
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"第 {attempt} 次失败：{e}")
            raise RuntimeError("重试耗尽")
        return wrapper
    return deco

calls = 0
@retry(3)
def flaky():
    global calls
    calls += 1
    if calls < 3:
        raise ValueError("网络抖动")
    return "ok"

print(flaky())

第 1 次失败：网络抖动
第 2 次失败：网络抖动
ok


## 六、几个容易踩的坑

**1. 循环里的晚绑定。** 闭包保存的是「变量的引用」，而不是「当时的值」。循环里创建的多个闭包共享同一个循环变量，等真正调用时循环早已结束，取到的是变量的最终值。

In [9]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])     # [2, 2, 2]，不是 [0, 1, 2]！

# 修复：用默认参数在「定义时」把值固化进每个闭包
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])     # [0, 1, 2]

[2, 2, 2]
[0, 1, 2]


**2. 忘写 `nonlocal`。** 在闭包里对外层变量赋值却没有 `nonlocal` 声明，Python 会把它当成新的局部变量，读值时触发 `UnboundLocalError`。判断规则：**赋值就需要 `nonlocal`，只读不需要**。

In [10]:
def broken_counter():
    count = 0
    def inc():
        count += 1              # 没有 nonlocal：赋值把 count 变成了局部变量
        return count
    return inc

try:
    broken_counter()()
except UnboundLocalError as e:
    print("报错了：", e)

报错了： cannot access local variable 'count' where it is not associated with a value


**3. 装饰后函数「换了身份」。** `add = log_calls(add)` 之后，`add.__name__` 变成了 `wrapper`，docstring 也丢了，会给调试和文档工具带来困扰。标准做法是用 `functools.wraps` 把原函数的元信息拷贝到 wrapper 上。

In [11]:
import functools

def log_calls(func):
    @functools.wraps(func)      # 把 __name__、__doc__ 等元信息拷给 wrapper
    def wrapper(*args, **kwargs):
        print(f"调用 {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def add(a, b):
    """两数相加。"""
    return a + b

print(add.__name__)             # add（没有 functools.wraps 时是 wrapper）
print(add.__doc__)              # 两数相加。

add
两数相加。


## 七、小练习

先自己动手，再对照后面的参考答案：

1. 写一个工厂函数 `make_power(n)`，返回一个计算 `x ** n` 的函数；用它生成 `square` 和 `cube`。
2. 写一个 `make_accumulator()`：返回的函数每被调用一次，就把参数累加进总额并返回当前总和（提示：`nonlocal`）。
3. 有 `students = [("小明", 88), ("小红", 95), ("小刚", 82)]`，用 `sorted` 加 `key` 按分数从高到低排序。
4. 写一个装饰器 `count_calls`，给被装饰的函数附加一个 `calls` 属性，记录它被调用的次数。
5. 修复 `[lambda: i for i in range(3)]`，让三个函数分别返回 0、1、2，并解释原来的输出为什么是 `[2, 2, 2]`。

In [12]:
# 1. 工厂函数
def make_power(n):
    def power(x):
        return x ** n
    return power

square, cube = make_power(2), make_power(3)
print(square(5), cube(2))               # 25 8

# 2. 带状态的闭包
def make_accumulator():
    total = 0
    def add(x):
        nonlocal total
        total += x
        return total
    return add

acc = make_accumulator()
print(acc(10), acc(5), acc(1))          # 10 15 16

# 3. 高阶函数排序
students = [("小明", 88), ("小红", 95), ("小刚", 82)]
print(sorted(students, key=lambda s: s[1], reverse=True))
# [('小红', 95), ('小明', 88), ('小刚', 82)]

# 4. 装饰器附加属性
import functools

def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def greet(name):
    return f"你好，{name}"

greet("小明")
greet("小红")
print(greet.calls)                      # 2

# 5. 固化循环变量
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])             # [0, 1, 2]

25 8
10 15 16
[('小红', 95), ('小明', 88), ('小刚', 82)]
2
[0, 1, 2]


## 总结

- `def` 是语句，函数可以嵌套定义；内层函数沿 **LEGB**（Local → Enclosing → Global → Builtin）向外层查找变量。
- **闭包 = 内层函数 + 捕获的自由变量**：函数返回后，外层变量经由 `__closure__` 继续存活；每次外层调用产生一个独立的环境。
- 在闭包内给外层变量**赋值**需要 `nonlocal` 声明，只读不需要。
- **高阶函数**是「接收函数或返回函数」的函数：`sorted(key=...)`、`map`、`filter` 是接收方向，`make_multiplier` 这类工厂函数是返回方向。
- **装饰器**是「接收函数、返回新函数」的高阶函数，`@deco` 等价于 `f = deco(f)`；带参数的装饰器是三层嵌套；记得用 `functools.wraps` 保住元信息。
- 循环里创建闭包要小心**晚绑定**：闭包记住的是变量本身，不是定义时的值。